# Phase 5b: LLM-as-judge evaluation

Stream B: tone alignment, accuracy, and grounding for baseline vs hybrid-RAG (`phase4b` JSONL outputs).

**Pipeline:** `phase4b` → `phase5a_ragas_eval.ipynb` → this notebook → `phase5c_pca_kmeans.R` → `phase5d_cross_stream_eval.ipynb`.

**Outputs:** `data/06_evaluation/llm_judge_*_v14_llm_judge_notebook.csv`, `llm_judge_hypothesis_tests_*.csv`, `llm_judge_tone_distribution_table.csv`, and `data/06_evaluation/figures/5b_llm_judge/v14_llm_judge_notebook/*.png`.

#Composer through a Cursor IDE guided the development of this script


In [ ]:
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from openai import OpenAI

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 140
load_dotenv()


In [ ]:
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "scripts" else Path.cwd().resolve()
import sys
SCRIPT_DIR = BASE_DIR / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
from pipeline_constants import (
    BASELINE_JSONL,
    HYBRID_JSONL,
    INFERENCE_TAG,
    LLM_JUDGE_EVAL_TAG,
)

BASELINE_INPUT = BASELINE_JSONL
HYBRID_INPUT = HYBRID_JSONL
OUTPUT_DIR = BASE_DIR / "data" / "06_evaluation"

TAG = LLM_JUDGE_EVAL_TAG
PROMPT_VERSION = INFERENCE_TAG
JUDGE_MODEL = "gpt-4o-mini"
MAX_RETRIES = 5
SLEEP_SECONDS = 0.15

RUN_JUDGE = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
per_row_path = OUTPUT_DIR / f"llm_judge_per_row_{TAG}.csv"
summary_path = OUTPUT_DIR / f"llm_judge_summary_{TAG}.csv"

assert BASELINE_INPUT.exists(), f"Missing: {BASELINE_INPUT}"
assert HYBRID_INPUT.exists(), f"Missing: {HYBRID_INPUT}"
assert INFERENCE_TAG in BASELINE_INPUT.name and INFERENCE_TAG in HYBRID_INPUT.name, (
    f"Expected {INFERENCE_TAG} JSONL paths; got baseline={BASELINE_INPUT.name} hybrid={HYBRID_INPUT.name}"
)

print(f"TAG={TAG} PROMPT_VERSION={PROMPT_VERSION} RUN_JUDGE={RUN_JUDGE}")
print(f"Output CSVs: {per_row_path.name}, {summary_path.name}")



## 1. Helpers, preflight, and judge rubric

Run preflight before setting `RUN_JUDGE = True`. The `judge_one` function below contains the full evaluation rubric (no API calls until §2).


In [ ]:
def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def extract_answer_only(text: str) -> str:
    if not isinstance(text, str):
        return ""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    for ln in lines:
        if ln.lower().startswith("answer:"):
            return ln.split(":", 1)[1].strip()
    return text.strip()

def build_reference_context_map(hybrid_rows):
    return {r.get("id"): r.get("retrieved_context", "") for r in hybrid_rows}

def judge_one(client, model: str, row: dict, reference_context: str, max_retries: int):
    system_prompt = (
        "You are an expert evaluator for UK SEND educational QA outputs. "
        "Evaluate one response against the question and provided context. "
        "Return strict JSON only."
    )
    model_response = extract_answer_only(row.get("ai_response", "")) or row.get("ai_response", "")
    user_prompt = f"""
Evaluate this response.

QUESTION:
{row.get('prompt', '')}

MODEL RESPONSE:
{model_response}

REFERENCE CONTEXT (for factual checking):
{reference_context}

Scoring rules:

Evaluate for a UK mainstream SENCo / school leader audience. Third-person, staff-facing 
answers are expected. Do not penalise calm, evidence-aware wording if it remains 
collegial and respectful.

--- TONE ---

1) tone_label: exactly one of ["supportive","neutral","directive"].

   supportive — Collegial SENCo voice that helps a practitioner feel equipped, not judged.
   Typical features (any combination):
   - Practical, respectful guidance oriented to pupils/staff/context
   - Non-blaming language; avoids shaming pupils, families, or colleagues
   - May use empathic framing OR calm professional clarity (both count as supportive)
   - Evidence-cautious phrasing is STILL supportive if collegial (e.g. "The evidence 
     suggests…", "Schools may consider…") and not dismissive
   Examples: actionable steps; balanced acknowledgement of difficulty; scope stated 
   professionally ("This depends on individual need…").

   neutral — Generic or flat instructional prose with weak SENCo character.
   Typical features:
   - Textbook-like, vague, or abstract; could apply to any sector
   - Little sense of UK SEND / mainstream inclusion context
   - Neither clearly helpful/collegial nor clearly harmful — but low practitioner warmth 
     or specificity
   NOT neutral: appropriately professional evidence-summary tone that is respectful 
   and staff-facing (label that supportive instead).

   directive — Inappropriately commanding, cold, or blaming.
   Typical features:
   - Harsh orders, blame toward child/family/staff, or alarmist certainty
   - "Must/always/never" without nuance where nuance is required
   - Dismissive refusal without collegial framing ("Just do X" / "This is wrong")

2) tone_alignment_score: integer 1-5 (must align with tone_label).

   5 — Exemplary UK SENCo professional tone: clear, respectful, non-judgmental, 
       practitioner-facing. Includes EITHER empathic supportive phrasing OR calm 
       evidence-informed professionalism when the question is factual/strategic.
       Appropriate hedging or scope limits do not reduce score if wording stays collegial.

   4 — Generally strong professional tone with minor flatness or slight vagueness; 
       still clearly appropriate for a SENCo advising colleagues.
       Leads with practical guidance in the first 1–2 sentences.

   3 — Neutral instructional tone: understandable but generic, thin on practitioner 
       value, or inconsistently professional (matches tone_label=neutral in most cases).
       Generic textbook prose with no UK SEND / mainstream-school framing.

   2 — Noticeably unsupportive: curt, unsympathetic, overly rigid, or partially blaming; 
       some usable content may remain.

   1 — Clearly directive in a harmful sense: dismissive, blaming, abrupt, or 
       inappropriate for sensitive SEND contexts (matches tone_label=directive).

   Mapping guide:
   - tone_label=supportive  → usually tone_alignment_score 4 or 5
   - tone_label=neutral     → usually tone_alignment_score 3 (sometimes 2 if very thin)
   - tone_label=directive   → usually tone_alignment_score 1 or 2

--- ACCURACY (factual usefulness to the QUESTION) ---

3) accuracy_score: integer 1-5.
   Score whether the answer helps the SENCo address the specific question asked.
   Use REFERENCE CONTEXT to check facts where relevant, but accuracy is about 
   question relevance and usefulness — not identical to grounding.

   5 — Directly answers the question with accurate, specific, practitioner-useful 
       content. A SENCo could apply or adapt this advice without major gaps.
       For "how do I…" questions: includes concrete strategies or clear next steps.
       For "what is / what are…" questions: correct explanation tailored to UK SEND.

   4 — Substantially accurate and useful with minor omissions, slight imprecision, 
       or missing one secondary detail; core guidance still sound.

   3 — Partly accurate or partially on-topic: some useful points but incomplete, 
       tangential, or too generic to act on confidently.
       Meta-statements alone ("evidence is mixed") without helping the posed question 
       usually belong here unless the question explicitly asks for evidence limits.

   2 — Mostly unhelpful: significant inaccuracies, wrong focus, or largely fails 
       the question while containing a few valid fragments.

   1 — Inaccurate, misleading, unsafe, or irrelevant to the question.

   Accuracy distinctions (same logic as tone):
   - Do NOT score 5 for polished tone alone if the question is not answered.
   - Appropriate scope/limit statements can still score 4-5 on accuracy IF they 
     honestly address what can/cannot be concluded from the context and guide next steps.
   - Do NOT conflate "brief" with "inaccurate" when the question is narrow and 
     the answer is correct.

--- GROUNDING ---

4) grounding_score: integer 1-5 for fidelity to REFERENCE CONTEXT.
   5 — Claims well supported by context; no clear invention beyond it.
   4 — Mostly supported; minor extrapolation or generalisation.
   3 — Mixed: some supported claims, some unsupported or under-specified.
   2 — Largely unsupported by context or over-claims beyond it.
   1 — Contradicts context or fabricates detail not present.
   Note: For baseline rows, context may be empty; score grounding on whether the 
   answer avoids presenting unsupported specifics as if cited from evidence.

5) concise_rationale: one short sentence citing the main reason for tone and accuracy scores.

Return JSON with keys exactly:
tone_label, tone_alignment_score, accuracy_score, grounding_score, concise_rationale
"""

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=0.0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            data = json.loads(resp.choices[0].message.content)
            return {
                "tone_label": str(data.get("tone_label", "neutral")).strip().lower(),
                "tone_alignment_score": int(data.get("tone_alignment_score", 3)),
                "accuracy_score": int(data.get("accuracy_score", 3)),
                "grounding_score": int(data.get("grounding_score", 3)),
                "concise_rationale": str(data.get("concise_rationale", "")).strip(),
            }
        except Exception as exc:
            last_err = exc
            time.sleep(min(2 ** attempt, 20))
    raise RuntimeError(f"Judge failed after retries: {last_err}")

def summarise(df: pd.DataFrame, dataset_name: str):
    sub = df[df["dataset"] == dataset_name]
    tone_counts = sub["tone_label"].value_counts().to_dict()
    return {
        "dataset": dataset_name,
        "rows": int(len(sub)),
        "tone_alignment_mean": round(float(sub["tone_alignment_score"].mean()), 4) if len(sub) else 0.0,
        "accuracy_mean": round(float(sub["accuracy_score"].mean()), 4) if len(sub) else 0.0,
        "grounding_mean": round(float(sub["grounding_score"].mean()), 4) if len(sub) else 0.0,
        "supportive_count": int(tone_counts.get("supportive", 0)),
        "neutral_count": int(tone_counts.get("neutral", 0)),
        "directive_count": int(tone_counts.get("directive", 0)),
    }


In [ ]:
def run_preflight_checks() -> None:
    baseline_rows = load_jsonl(BASELINE_INPUT)
    hybrid_rows = load_jsonl(HYBRID_INPUT)
    assert len(baseline_rows) == 250, f"Expected 250 baseline rows, got {len(baseline_rows)}"
    assert len(hybrid_rows) == 250, f"Expected 250 hybrid rows, got {len(hybrid_rows)}"

    b_ids = [r.get("id") for r in baseline_rows]
    h_ids = [r.get("id") for r in hybrid_rows]
    assert b_ids == h_ids, "Baseline and hybrid id order differ"

    sample = baseline_rows[0]
    for key in ("id", "prompt", "ai_response", "category"):
        assert key in sample, f"Missing key in JSONL: {key}"
    ans = extract_answer_only(sample.get("ai_response", ""))
    assert ans, "extract_answer_only returned empty on first baseline row"

    api_key = (os.getenv("OPENAI_API_KEY") or "").strip()
    if not api_key or api_key.startswith("your-"):
        raise ValueError("OPENAI_API_KEY missing or placeholder in .env")

    context_map = build_reference_context_map(hybrid_rows)
    empty_ctx = sum(1 for c in context_map.values() if not (c or "").strip())

    cached = per_row_path.exists() and summary_path.exists()
    print(
        f"Preflight OK | tag={TAG} rows=250 | empty_ctx={empty_ctx} | "
        f"RUN_JUDGE={RUN_JUDGE} cached_csvs={cached}"
    )

run_preflight_checks()



## 2. Run judge (or load cached CSVs)

Set `RUN_JUDGE = True` to score all 500 responses. Requires `OPENAI_API_KEY` in `.env`.


In [ ]:
if RUN_JUDGE:
    baseline_rows = load_jsonl(BASELINE_INPUT)
    hybrid_rows = load_jsonl(HYBRID_INPUT)
    context_map = build_reference_context_map(hybrid_rows)

    client = OpenAI()
    all_rows = []
    combined = [("baseline", r) for r in baseline_rows] + [("hybrid_rag", r) for r in hybrid_rows]

    for i, (dataset_name, row) in enumerate(combined, start=1):
        ref_ctx = context_map.get(row.get("id"), "")
        judged = judge_one(client, JUDGE_MODEL, row, ref_ctx, MAX_RETRIES)
        all_rows.append({
            "dataset": dataset_name,
            "id": row.get("id"),
            "category": row.get("category"),
            "prompt": row.get("prompt"),
            "answer_text": extract_answer_only(row.get("ai_response", "")),
            **judged,
        })
        if i % 10 == 0 or i == len(combined):
            print(f"[{i}/{len(combined)}] judged")
        time.sleep(SLEEP_SECONDS)

    per_row_df = pd.DataFrame(all_rows)
    summary_df = pd.DataFrame([
        summarise(per_row_df, "baseline"),
        summarise(per_row_df, "hybrid_rag"),
    ])
    per_row_df.to_csv(per_row_path, index=False)
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved: {per_row_path}")
    print(f"Saved: {summary_path}")
else:
    if not per_row_path.exists() or not summary_path.exists():
        raise FileNotFoundError(
            f"Missing judge CSVs. Set RUN_JUDGE=True or run evaluation first: {per_row_path}, {summary_path}"
        )
    per_row_df = pd.read_csv(per_row_path)
    summary_df = pd.read_csv(summary_path)
    print(f"Loaded cached judge CSVs (RUN_JUDGE=False): {per_row_path.name}, {summary_path.name}")


## 3. Summary


In [ ]:
summary_df


## 4. Figures


In [ ]:
def plot_overall_metric_bars(summary_df: pd.DataFrame, fig_dir: Path):
    mdf = summary_df.melt(
        id_vars=["dataset"],
        value_vars=["tone_alignment_mean", "accuracy_mean", "grounding_mean"],
        var_name="metric",
        value_name="score",
    )
    plt.figure(figsize=(8, 5))
    ax = sns.barplot(data=mdf, x="metric", y="score", hue="dataset", errorbar=None)
    ax.set_ylim(0, 5)
    ax.set_title("LLM Judge Mean Scores by Dataset")
    ax.set_xlabel("")
    ax.set_ylabel("Score (1-5)")
    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%.2f",
            label_type="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(fig_dir / "5b_overall_metric_means.png", bbox_inches="tight")
    plt.close()


def plot_tone_distribution(summary_df: pd.DataFrame, fig_dir: Path):
    tone_cols = ["supportive_count", "neutral_count", "directive_count"]
    tone_df = summary_df[["dataset"] + tone_cols].set_index("dataset")
    ax = tone_df.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="Set2")
    ax.set_title("Tone Label Distribution by Dataset")
    ax.set_ylabel("Count")

    # Add labels for each segment and total count at the end of each bar
    for patch in ax.patches:
        height = patch.get_height()
        if height > 0:
            x = patch.get_x() + patch.get_width() / 2
            y = patch.get_y() + height / 2
            ax.text(x, y, f"{int(height)}", ha='center', va='center', fontsize=9, color='white', fontweight='bold')

    totals = tone_df.sum(axis=1)
    for i, total in enumerate(totals):
        # Position total slightly above the stacked bar
        x = i
        y = total
        ax.text(x, y + max(totals) * 0.01, f"{int(total)}", ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(fig_dir / "5b_tone_distribution.png", bbox_inches="tight")
    plt.close()


def plot_category_metric_bars(per_row_df: pd.DataFrame, fig_dir: Path):
    cat_means = (
        per_row_df.groupby(["category", "dataset"], as_index=False)[
            ["tone_alignment_score", "accuracy_score", "grounding_score"]
        ].mean(numeric_only=True)
    )
    mdf = cat_means.melt(
        id_vars=["category", "dataset"],
        value_vars=["tone_alignment_score", "accuracy_score", "grounding_score"],
        var_name="metric",
        value_name="score",
    )
    g = sns.catplot(
        data=mdf,
        kind="bar",
        x="category",
        y="score",
        hue="dataset",
        col="metric",
        errorbar=("ci", 95),
        height=5,
        aspect=1.3,
        sharey=True,
    )
    g.set_xticklabels(rotation=30, ha="right")
    g.set_ylabels("Score (1-5)")
    g.fig.suptitle("LLM Judge Scores by Category (Stratified)", y=1.03)
    g.savefig(fig_dir / "5b_category_metric_bars.png", bbox_inches="tight")
    plt.close("all")


def plot_category_delta_heatmap(per_row_df: pd.DataFrame, fig_dir: Path):
    cat_means = (
        per_row_df.groupby(["category", "dataset"], as_index=False)[
            ["tone_alignment_score", "accuracy_score", "grounding_score"]
        ].mean(numeric_only=True)
    )
    judge_metrics = ["tone_alignment_score", "accuracy_score", "grounding_score"]
    b = cat_means[cat_means["dataset"] == "baseline"].set_index("category")
    h = cat_means[cat_means["dataset"] == "hybrid_rag"].set_index("category")
    delta = h[judge_metrics] - b[judge_metrics]
    delta = delta.sort_index()
    overall = per_row_df.groupby("dataset")[judge_metrics].mean(numeric_only=True)
    delta.loc["Overall Delta"] = overall.loc["hybrid_rag"] - overall.loc["baseline"]

    fig, ax = plt.subplots(figsize=(8, 5.4))
    sns.heatmap(
        delta,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        center=0,
        cbar_kws={"label": "Hybrid - Baseline"},
        ax=ax,
    )
    ax.set_title("Category-Level LLM Judge Delta")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Category")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(fig_dir / "5b_category_delta_heatmap.png", bbox_inches="tight")
    plt.close()


def save_hypothesis_tests_table(per_row_df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    """Paired Wilcoxon tests: export CSV table only (no chart)."""
    from scipy.stats import wilcoxon

    factors = ["tone_alignment_score", "accuracy_score", "grounding_score"]
    labels = {
        "tone_alignment_score": "Tone Alignment",
        "accuracy_score": "Accuracy",
        "grounding_score": "Grounding",
    }
    rows = []
    paired = per_row_df.pivot(index="id", columns="dataset", values=factors)
    for factor in factors:
        baseline = paired[(factor, "baseline")].astype(float)
        hybrid = paired[(factor, "hybrid_rag")].astype(float)
        baseline, hybrid = baseline.align(hybrid, join="inner")
        _stat, p_value = wilcoxon(hybrid, baseline, zero_method="wilcox", correction=False)
        rows.append({
            "metric": labels[factor],
            "baseline_mean": round(float(baseline.mean()), 4),
            "hybrid_mean": round(float(hybrid.mean()), 4),
            "mean_diff_hybrid_minus_baseline": round(float(hybrid.mean() - baseline.mean()), 4),
            "p_value": float(p_value),
        })

    hypothesis_df = pd.DataFrame(rows)
    hypothesis_path = output_dir / f"llm_judge_hypothesis_tests_{TAG}.csv"
    hypothesis_df.to_csv(hypothesis_path, index=False)
    print(f"Saved hypothesis tests table: {hypothesis_path}")
    return hypothesis_df


def save_5b_figures(summary_df: pd.DataFrame, per_row_df: pd.DataFrame, fig_dir: Path):
    plot_overall_metric_bars(summary_df, fig_dir)
    plot_tone_distribution(summary_df, fig_dir)
    plot_category_metric_bars(per_row_df, fig_dir)
    plot_category_delta_heatmap(per_row_df, fig_dir)
    save_hypothesis_tests_table(per_row_df, OUTPUT_DIR)
    tone_table = summary_df.set_index("dataset")[
        ["supportive_count", "neutral_count", "directive_count"]
    ].T
    tone_table.index = ["supportive", "neutral", "directive"]
    tone_table.columns = ["baseline", "hybrid_rag"]
    tone_table.to_csv(OUTPUT_DIR / "llm_judge_tone_distribution_table.csv")
    print(f"5b figures saved to: {fig_dir}")


In [ ]:
fig_dir = OUTPUT_DIR / "figures" / "5b_llm_judge" / TAG
fig_dir.mkdir(parents=True, exist_ok=True)

save_5b_figures(summary_df, per_row_df, fig_dir)
